# 01 · Data Cleaning: Amazon India Store

**Project:** End-to-end sales analysis (Python → SQL Server → Power BI)
**Input:** `raw_csv/*.csv` (7 tables exported from the MySQL dump)
**Output:** `clean_csv/*.csv`, `reports/data_quality_log.csv`, `reports/column_changes.csv`

**Cleaning principles used here**
1. **Never silently delete data.** Where a value is suspicious I add a `flag` column and keep the original.
2. **Recompute, don't trust.** Pre-aggregated columns (customer `total_orders`, seller `total_sales_inr`, ...) are checked against the transaction tables and replaced by recomputed values. The originals are kept as `*_reported`.
3. **Every change is logged** (table, issue, rows affected, action, severity) so the cleaning is auditable.
4. Outputs are **SQL Server-ready**: ISO dates, `Yes/No` → `0/1` (BIT), no mixed types.

In [1]:
from pathlib import Path
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 170)

RAW, CLEAN, REPORTS = Path("../raw_csv"), Path("../clean_csv"), Path("../reports")
CLEAN.mkdir(exist_ok=True); REPORTS.mkdir(exist_ok=True)

TABLES = ["sellers","customers","products","orders","returns","transactions","reviews"]
raw = {t: pd.read_csv(RAW/f"{t}.csv") for t in TABLES}
sellers, customers, products, orders, returns, transactions, reviews = [raw[t].copy() for t in TABLES]
frames = dict(sellers=sellers, customers=customers, products=products, orders=orders,
              returns=returns, transactions=transactions, reviews=reviews)

# --- audit log helper -------------------------------------------------------
log = []
def note(table, issue, rows, action, severity):
    log.append(dict(table=table, issue=issue, rows_affected=int(rows), action=action, severity=severity))

audit = pd.DataFrame({t: {"rows": len(d), "cols": d.shape[1], "null_cells": int(d.isna().sum().sum()),
                          "duplicate_rows": int(d.duplicated().sum())} for t, d in raw.items()}).T
audit

,rows,cols,null_cells,duplicate_rows
sellers,75,18,0,0
customers,320,15,0,0
products,180,19,0,0
orders,2100,21,1084,0
returns,420,13,0,0
transactions,2100,15,6495,0
reviews,1850,14,0,0


## 1 · Raw audit: structure & keys

In [2]:
pks = dict(sellers="seller_id", customers="customer_id", products="product_id", orders="order_id",
           returns="return_id", transactions="transaction_id", reviews="review_id")
for t, k in pks.items():
    dup = frames[t][k].duplicated().sum()
    print(f"{t:<13} PK {k:<15} duplicates: {dup}")
    assert dup == 0

print("\nDuplicate business keys:  email:", customers.email.duplicated().sum(),
      "| mobile:", customers.mobile.duplicated().sum(), "| ASIN:", products.asin.duplicated().sum())
print("\nNull cells by column (only columns with nulls):")
for t, d in raw.items():
    n = d.isna().sum(); n = n[n > 0]
    if len(n): print(f"  {t}: {n.to_dict()}")

sellers       PK seller_id       duplicates: 0
customers     PK customer_id     duplicates: 0
products      PK product_id      duplicates: 0
orders        PK order_id        duplicates: 0
returns       PK return_id       duplicates: 0
transactions  PK transaction_id  duplicates: 0
reviews       PK review_id       duplicates: 0

Duplicate business keys:  email: 0 | mobile: 0 | ASIN: 0

Null cells by column (only columns with nulls):
  orders: {'delivery_date': 368, 'promo_applied': 716}
  transactions: {'bank_name': 1045, 'upi_id': 1822, 'failure_reason': 1948, 'linked_return_id': 1680}


## 2 · Standardise types & formats
* All dates were stored as text → `datetime`
* Whitespace trimmed on every text column
* Mobile numbers `+91 XXXXXXXXXX` (with a space) → `+91XXXXXXXXXX`
* `Yes/No` columns → `0/1` flags (maps to `BIT` in SQL Server)

In [3]:
DATE_COLS = {"sellers":["joined_date"], "customers":["account_created"], "products":["launch_date"],
             "orders":["order_date","ship_date","delivery_date"], "returns":["return_request_date"],
             "transactions":["transaction_date"], "reviews":["review_date"]}
n_dates = 0
for t, cols in DATE_COLS.items():
    for c in cols:
        before = frames[t][c].isna().sum()
        frames[t][c] = pd.to_datetime(frames[t][c], errors="coerce")
        assert frames[t][c].isna().sum() == before, f"unparseable dates in {t}.{c}"
        n_dates += 1
note("ALL", "Dates stored as text (VARCHAR)", n_dates, f"Converted {n_dates} columns to datetime; 0 unparseable values", "Medium")

# trim whitespace on text columns
trimmed = 0
for d in frames.values():
    for c in d.columns:
        if not pd.api.types.is_numeric_dtype(d[c]) and not pd.api.types.is_datetime64_any_dtype(d[c]):
            new = d[c].str.strip(); trimmed += int((new.fillna("") != d[c].fillna("")).sum()); d[c] = new
note("ALL", "Leading/trailing whitespace in text", trimmed, "Trimmed (none found)" if trimmed == 0 else "Trimmed", "Low")

# mobile format
digits = customers.mobile.str.replace(r"\D", "", regex=True)
assert digits.str.len().eq(12).all() and digits.str.startswith("91").all()
new_mobile = "+91" + digits.str[-10:]
note("customers", "Mobile has a space (+91 XXXXXXXXXX), fails +91XXXXXXXXXX pattern", (customers.mobile != new_mobile).sum(),
     "Normalised to +91XXXXXXXXXX", "Low")
customers["mobile"] = new_mobile

# Yes/No -> 0/1
def to_flag(s):
    out = s.map({"Yes": 1, "No": 0}); assert out.notna().all(), s.name; return out.astype(int)
for t, cols in {"sellers":["brand_registered"], "customers":["kyc_verified","gst_registered"], "products":["is_prime"],
                "orders":["is_business_order"], "returns":["replacement_issued"], "transactions":["is_refund"],
                "reviews":["verified_purchase","vine_review","image_attached","seller_response"]}.items():
    for c in cols: frames[t][c] = to_flag(frames[t][c])
customers["is_prime_member"] = (customers.prime_member == "Prime").astype(int)
customers = customers.drop(columns="prime_member"); frames["customers"] = customers
note("ALL", "Yes/No text columns", 11, "Converted to 0/1 flags (SQL Server BIT)", "Low")
customers[["customer_id","mobile","is_prime_member","kyc_verified"]].head(3)

,customer_id,mobile,is_prime_member,kyc_verified
0,INCUST00001,+917883838371,0,1
1,INCUST00002,+918745040691,0,1
2,INCUST00003,+919954673466,0,1


## 3 · Orders: reconcile status, returns and payments

**Problem:** `orders.order_status` and the `returns` table disagree.
* 375 of 420 return records sit on orders whose status is *Delivered*
* 106 of 151 orders marked *Returned* have **no** return record

**Decision:** treat the **`returns` table as the source of truth** (it carries reason, refund and dates) and derive clean fields on `orders`.
Nothing is overwritten: the raw `order_status` is kept.

| New column | Meaning |
|---|---|
| `fulfilment_status` | Delivery stage only (*Returned* collapsed into *Delivered*: an item can't be returned without being delivered) |
| `has_return_record` | Order has a row in `returns` |
| `is_returned` | Return requested = has return record **or** raw status is *Returned* |
| `return_evidence` | `Return record` / `Status only` / `No Return` |
| `return_outcome` | `Completed` (Refunded/Exchanged) · `In progress` · `Rejected` · `No Return` |
| `refund_amount_inr` | Cash refunded (only `return_status = Refunded`) |
| `net_revenue_inr` | Order total (0 if cancelled) minus refunds: the revenue measure for the dashboard |

In [4]:
o = orders
rmap = returns.set_index("order_id")
o["has_return_record"] = o.order_id.isin(returns.order_id).astype(int)
o["return_status"] = o.order_id.map(rmap.return_status)
o["return_outcome"] = np.select(
    [o.return_status.isin(["Refunded","Exchanged"]), o.return_status.eq("Rejected"), o.return_status.isin(["Under Review","Pickup Scheduled"])],
    ["Completed","Rejected","In progress"], default="No Return")
o["returned_status_no_record"] = ((o.order_status == "Returned") & (o.has_return_record == 0)).astype(int)
o["is_returned"] = ((o.has_return_record == 1) | (o.order_status == "Returned")).astype(int)
o["return_evidence"] = np.select([o.has_return_record == 1, o.returned_status_no_record == 1], ["Return record","Status only"], default="No Return")
o["fulfilment_status"] = o.order_status.replace({"Returned": "Delivered"})
o["refund_amount_inr"] = o.order_id.map(rmap.refund_amount_inr).where(o.return_status == "Refunded", 0).fillna(0)
o["is_cancelled"] = (o.order_status == "Cancelled").astype(int)
o["net_revenue_inr"] = (o.order_total_inr * (1 - o.is_cancelled) - o.refund_amount_inr).round(2)

note("orders", "Return records on orders whose status is 'Delivered' (status ≠ returns table)",
     ((o.has_return_record == 1) & (o.order_status == "Delivered")).sum(),
     "Kept raw status; added has_return_record / is_returned / return_outcome from returns table", "High")
note("orders", "Orders with status 'Returned' but NO return record", o.returned_status_no_record.sum(),
     "Flagged returned_status_no_record; counted as returned with return_evidence='Status only'; refund unknown (0)", "High")

# ---- other order-level checks ---------------------------------------------------
# a) total formula
calc = o.subtotal_inr + o.shipping_fee_inr + o.gst_inr - o.discount_inr
assert ((calc - o.order_total_inr).abs() < 1).all(); print("order_total formula holds for all rows")

# b) promo vs discount (compute BEFORE filling nulls)
o["discount_without_promo"] = ((o.discount_inr > 0) & o.promo_applied.isna()).astype(int)
o["promo_without_discount"] = ((o.discount_inr == 0) & o.promo_applied.notna()).astype(int)
note("orders", "Discount given but no promo code", o.discount_without_promo.sum(), "Flagged discount_without_promo", "Medium")
note("orders", "Promo code recorded but discount = 0", o.promo_without_discount.sum(), "Flagged promo_without_discount", "Medium")
note("orders", "promo_applied is NULL", o.promo_applied.isna().sum(), "Filled with 'No Promo' (NULL means no promo used)", "Low")
o["promo_applied"] = o.promo_applied.fillna("No Promo")

# c) shipping_type contains a status value
note("orders", "shipping_type = 'Out for Delivery' is an order status, not a shipping speed", (o.shipping_type == "Out for Delivery").sum(),
     "Created shipping_type_clean with value 'Unspecified'", "Medium")
o["shipping_type_clean"] = o.shipping_type.replace({"Out for Delivery": "Unspecified"})

# d) Prime delivery on non-Prime customers
prime = customers.set_index("customer_id").is_prime_member
o["prime_delivery_mismatch"] = ((o.shipping_type == "Prime Free Delivery") & (o.customer_id.map(prime) == 0)).astype(int)
note("orders", "Non-Prime customers given 'Prime Free Delivery'", o.prime_delivery_mismatch.sum(), "Flagged prime_delivery_mismatch", "Medium")

# e) ordered before product launch
launch = products.set_index("product_id").launch_date
o["order_before_launch"] = (o.order_date < o.product_id.map(launch)).astype(int)
note("orders", "Order placed before product's launch_date", o.order_before_launch.sum(), "Flagged order_before_launch (dates kept)", "Medium")

# f) ship date on orders that cannot have shipped yet
o["ship_date_on_unshipped"] = (o.order_status.isin(["Pending","Processing"]) & o.ship_date.notna()).astype(int)
note("orders", "ship_date populated while status is Pending/Processing", o.ship_date_on_unshipped.sum(), "Flagged ship_date_on_unshipped", "Medium")

# g) delivery_date nulls are legitimate?
bad = o[(o.delivery_date.isna()) & o.order_status.isin(["Delivered","Returned","Shipped","Out for Delivery"])]
note("orders", "delivery_date NULL", o.delivery_date.isna().sum(),
     f"Legitimate: all on Cancelled/Pending/Processing orders (violations: {len(bad)}); left NULL", "Low")
# shipped/out-for-delivery orders DO carry a delivery_date -> flag as it's a future/actual mix
o["delivery_date_on_in_transit"] = (o.order_status.isin(["Shipped","Out for Delivery"]) & o.delivery_date.notna()).astype(int)

# ---- derived time features ------------------------------------------------------
o["order_year"] = o.order_date.dt.year
o["order_quarter"] = o.order_date.dt.quarter
o["order_month"] = o.order_date.dt.to_period("M").dt.to_timestamp()
o["order_weekday"] = o.order_date.dt.day_name()
o["days_to_ship"] = (o.ship_date - o.order_date).dt.days
o["days_to_deliver"] = (o.delivery_date - o.order_date).dt.days
assert (o.days_to_ship.dropna() >= 0).all() and (o.days_to_deliver.dropna() >= 0).all()
note("orders", "Delivery time does not differ by shipping_type (Next Day ≈ Economy)", 0,
     "Not a cleaning issue; carried into diagnostic analysis as a finding", "Info")

o[["order_id","order_status","fulfilment_status","is_returned","return_evidence","return_outcome","refund_amount_inr","net_revenue_inr"]].sample(5, random_state=1)

order_total formula holds for all rows


,order_id,order_status,fulfilment_status,is_returned,return_evidence,return_outcome,refund_amount_inr,net_revenue_inr
1601,408-8010013-8715633,Processing,Processing,0,No Return,No Return,0.00,3717.43
1268,408-6677577-9167108,Out for Delivery,Out for Delivery,0,No Return,No Return,0.00,887.90
587,408-3655462-9177369,Shipped,Shipped,0,No Return,No Return,0.00,8549.91
950,408-5345824-4915847,Shipped,Shipped,0,No Return,No Return,0.00,27644.34
1613,408-8048991-9629264,Delivered,Delivered,1,Return record,Completed,13246.09,21.81


In [5]:
print("Order status vs. return evidence")
print(pd.crosstab(o.order_status, o.return_evidence, margins=True))
print("\nGross vs net revenue (INR): gross =", f"{o.loc[o.is_cancelled==0,'order_total_inr'].sum():,.0f}",
      "| refunds =", f"{o.refund_amount_inr.sum():,.0f}", "| net =", f"{o.net_revenue_inr.sum():,.0f}")

Order status vs. return evidence
return_evidence   No Return  Return record  Status only   All
order_status                                                 
Cancelled               176              0            0   176
Delivered               879            375            0  1254
Out for Delivery        116              0            0   116
Pending                  72              0            0    72
Processing              120              0            0   120
Returned                  0             45          106   151
Shipped                 211              0            0   211
All                    1574            420          106  2100

Gross vs net revenue (INR): gross = 18,214,813 | refunds = 2,031,661 | net = 16,183,152


## 4 · Returns
`days_to_return` turns out to be measured from **order date** (verified below), not delivery date, so it can't be used against `return_window_days`. I add a delivery-based measure and flag rule breaks.

In [6]:
r = returns.merge(orders[["order_id","order_date","delivery_date"]], on="order_id", how="left")
r = r.merge(products[["product_id","return_window_days"]], on="product_id", how="left")

assert (r.days_to_return == (r.return_request_date - r.order_date).dt.days).all()
note("returns", "days_to_return is measured from ORDER date (not delivery); misleading against return_window_days",
     len(r), "Renamed to days_from_order_to_return; added days_since_delivery", "Medium")
r["days_from_order_to_return"] = r.days_to_return
r["days_since_delivery"] = (r.return_request_date - r.delivery_date).dt.days
r["request_before_delivery"] = (r.days_since_delivery < 0).astype(int)
r["outside_return_window"] = (r.days_since_delivery > r.return_window_days).astype(int)
r["refund_pct_of_order"] = (r.refund_amount_inr / r.order_id.map(orders.set_index("order_id").order_total_inr) * 100).round(1)

note("returns", "Return requested BEFORE the order was delivered", r.request_before_delivery.sum(), "Flagged request_before_delivery", "Medium")
note("returns", "Return requested after return_window_days (measured from delivery)", r.outside_return_window.sum(), "Flagged outside_return_window", "Medium")
note("returns", "Refund > order total", (r.refund_pct_of_order > 100).sum(), "None found", "Info")

returns = r.drop(columns=["days_to_return","order_date","delivery_date","return_window_days"]); frames["returns"] = returns
print("Refund as % of order value:", returns.refund_pct_of_order.describe()[["min","50%","max"]].round(1).to_dict())
returns[["return_id","days_from_order_to_return","days_since_delivery","request_before_delivery","outside_return_window"]].head()

Refund as % of order value: {'min': 70.0, '50%': 84.6, 'max': 100.0}


,return_id,days_from_order_to_return,days_since_delivery,request_before_delivery,outside_return_window
0,INRET000001,13,7,0,0
1,INRET000002,21,20,0,1
2,INRET000003,19,14,0,1
3,INRET000004,14,5,0,0
4,INRET000005,30,23,0,1


## 5 · Transactions: payment status vs order status

`transactions` is 1-to-1 with `orders`. I check whether the payment story is coherent with the order story and label each conflict.

In [7]:
tx = transactions.merge(orders[["order_id","order_status","has_return_record","return_status"]], on="order_id", how="left")
fulfilled = ["Delivered","Returned","Shipped","Out for Delivery"]
tx["payment_conflict_type"] = np.select(
    [ (tx.transaction_status == "Failed")  & tx.order_status.isin(fulfilled),
      (tx.transaction_status == "Success") & (tx.order_status == "Cancelled"),
      (tx.transaction_status == "Pending") & tx.order_status.isin(["Delivered","Returned"]) & (tx.payment_method != "Cash on Delivery"),
      (tx.transaction_status == "Refunded") & (tx.has_return_record == 0),
      (tx.return_status == "Refunded") & (tx.transaction_status != "Refunded") ],
    ["Fulfilled despite failed payment", "Cancelled but payment captured", "Delivered but prepaid payment still pending",
     "Refund transaction without a return record", "Return refunded but transaction not marked Refunded"],
    default="No Conflict")
tx["payment_conflict"] = (tx.payment_conflict_type != "No Conflict").astype(int)
print(tx.payment_conflict_type.value_counts())
note("transactions", "Payment status contradicts order/return status", tx.payment_conflict.sum(),
     "Flagged payment_conflict + payment_conflict_type (kept as-is; needs business reconciliation)", "High")

# linked_return_id checks
refund_unlinked = tx[(tx.is_refund == 1) & tx.linked_return_id.isna()]
recoverable = refund_unlinked.order_id.isin(returns.order_id).sum()
note("transactions", "Refund transactions (is_refund=1) with NULL linked_return_id", len(refund_unlinked),
     f"Attempted recovery via order_id join: {recoverable} recoverable → left NULL (their orders have no return record)", "High")
linked_not_refund = tx[tx.linked_return_id.notna() & (tx.is_refund == 0)]
note("transactions", "Transactions linked to a return but is_refund = 0", len(linked_not_refund),
     "Kept; is_refund is the payment-side flag, return_status on returns is the business-side one", "Medium")
assert tx.linked_return_id.dropna().isin(returns.return_id).all()

# card / UPI completeness
note("transactions", "Credit/Debit card without bank_name", (tx.payment_method.isin(["Credit Card","Debit Card"]) & tx.bank_name.isna()).sum(), "None found", "Info")
note("transactions", "UPI without upi_id", ((tx.payment_method == "UPI") & tx.upi_id.isna()).sum(), "None found", "Info")
# amount vs order total
chk = tx.merge(orders[["order_id","order_total_inr"]], on="order_id")
note("transactions", "amount_inr ≠ order_total_inr", ((chk.amount_inr - chk.order_total_inr).abs() > 1).sum(), "None found (amount is the order amount, not the refund amount)", "Info")

transactions = tx.drop(columns=["order_status","has_return_record","return_status"]); frames["transactions"] = transactions

payment_conflict_type
No Conflict                                            1699
Return refunded but transaction not marked Refunded     153
Refund transaction without a return record              114
Fulfilled despite failed payment                         60
Delivered but prepaid payment still pending              43
Cancelled but payment captured                           31
Name: count, dtype: int64


## 6 · Customers: recompute stale aggregates
The reported `total_orders` (mean ≈ 41) and `lifetime_spend_inr` bear no relation to the order table (mean ≈ 7 orders per customer). I test this, then recompute.

In [8]:
act_orders = orders.groupby("customer_id").size()
act_spend  = orders[orders.is_cancelled == 0].groupby("customer_id").order_total_inr.sum()
first_order = orders.groupby("customer_id").order_date.min()
last_order  = orders.groupby("customer_id").order_date.max()

print("Correlation reported vs actual:  orders:", round(np.corrcoef(customers.total_orders, customers.customer_id.map(act_orders).fillna(0))[0,1], 3),
      "| spend:", round(np.corrcoef(customers.lifetime_spend_inr, customers.customer_id.map(act_spend).fillna(0))[0,1], 3))

customers = customers.rename(columns={"total_orders":"total_orders_reported", "lifetime_spend_inr":"lifetime_spend_reported_inr"})
customers["orders_actual"] = customers.customer_id.map(act_orders).fillna(0).astype(int)
customers["lifetime_spend_actual_inr"] = customers.customer_id.map(act_spend).fillna(0).round(2)
customers["first_order_date"] = customers.customer_id.map(first_order)
customers["last_order_date"] = customers.customer_id.map(last_order)
customers["has_orders"] = (customers.orders_actual > 0).astype(int)
note("customers", "total_orders / lifetime_spend_inr don't match the orders table (mean 41 vs 7 orders; corr ≈ 0)", len(customers),
     "Recomputed orders_actual & lifetime_spend_actual_inr (non-cancelled); originals kept as *_reported", "High")
note("customers", "Customers with no orders in the period", (customers.has_orders == 0).sum(), "Flagged has_orders = 0 (kept; matters for conversion analysis)", "Low")

late = customers.account_created > customers.first_order_date
customers["account_created_after_first_order"] = late.astype(int)
customers["account_created_clean"] = customers.account_created.where(~late, customers.first_order_date)
note("customers", "account_created is AFTER the customer's first order", late.sum(),
     "Flagged; account_created_clean = first order date for these rows", "Medium")

note("customers", "Gender values outside M/F", (~customers.gender.isin(["M","F"])).sum(), "None found", "Info")
note("customers", "Pincode not 6 digits", (~customers.pincode.astype(str).str.fullmatch(r"\d{6}")).sum(), "None found", "Info")
note("customers", "Email missing '@'", (~customers.email.str.contains("@")).sum(), "None found", "Info")
frames["customers"] = customers
customers[["customer_id","total_orders_reported","orders_actual","lifetime_spend_reported_inr","lifetime_spend_actual_inr"]].head()

Correlation reported vs actual:  orders: -0.092 | spend: -0.007


,customer_id,total_orders_reported,orders_actual,lifetime_spend_reported_inr,lifetime_spend_actual_inr
0,INCUST00001,49,15,28460.53,160457.31
1,INCUST00002,35,6,29687.37,20188.82
2,INCUST00003,44,1,712.83,1008.83
3,INCUST00004,68,15,17509.33,72434.39
4,INCUST00005,12,8,6021.63,11151.39


## 7 · Products: margin, discount and review consistency

In [9]:
p = products
p["margin_inr"] = (p.selling_price - p.cost_price).round(2)
p["margin_pct"] = (p.margin_inr / p.selling_price * 100).round(2)
p["negative_margin"] = (p.margin_inr < 0).astype(int)
note("products", "cost_price > selling_price (sold at a loss)", p.negative_margin.sum(), "Flagged negative_margin; margin_inr/margin_pct added", "Medium")

p["discount_pct_calc"] = ((1 - p.selling_price / p.mrp_inr) * 100).round(1)
p["discount_pct_mismatch"] = ((p.discount_pct_calc - p.discount_pct).abs() > 1).astype(int)
note("products", "discount_pct ≠ (1 - selling/MRP)", p.discount_pct_mismatch.sum(), "Added discount_pct_calc; flagged discount_pct_mismatch", "Medium")
note("products", "selling_price > MRP", (p.selling_price > p.mrp_inr).sum(), "None found", "Info")

rv = reviews.groupby("product_id").agg(review_count_actual=("review_id","count"), avg_review_rating=("rating","mean"))
p = p.rename(columns={"review_count":"review_count_reported"}).merge(rv, on="product_id", how="left")
p["review_count_actual"] = p.review_count_actual.fillna(0).astype(int)
p["avg_review_rating"] = p.avg_review_rating.round(2)
print("Correlation listing rating vs mean of review ratings:", round(np.corrcoef(p.rating, p.avg_review_rating.fillna(p.rating.mean()))[0,1], 3))
note("products", "review_count is a listing figure (≠ rows in reviews); rating ≠ mean review rating", len(p),
     "Kept both: review_count_reported/rating vs review_count_actual/avg_review_rating (reviews table is a sample)", "Medium")

p = p.merge(sellers[["seller_id","joined_date"]].rename(columns={"joined_date":"_sj"}), on="seller_id", how="left")
p["seller_joined_after_launch"] = (p._sj > p.launch_date).astype(int); p = p.drop(columns="_sj")
note("products", "Seller joined AFTER the product's launch_date", p.seller_joined_after_launch.sum(), "Flagged seller_joined_after_launch", "Low")
note("products", "weight_kg = 0 or > 50", ((p.weight_kg <= 0) | (p.weight_kg > 50)).sum(), "None found (range 0.31–11.92)", "Info")
frames["products"] = products = p
p[["product_id","selling_price","cost_price","margin_pct","negative_margin","discount_pct","discount_pct_calc"]].head()

Correlation listing rating vs mean of review ratings: -0.06


,product_id,selling_price,cost_price,margin_pct,negative_margin,discount_pct,discount_pct_calc
0,IN-PROD-0001,2661.97,1614,39.37,0,0,4.9
1,IN-PROD-0002,1605.98,838,47.82,0,10,10.7
2,IN-PROD-0003,5630.00,3166,43.77,0,0,6.2
3,IN-PROD-0004,521.85,285,45.39,0,10,12.9
4,IN-PROD-0005,320.47,253,21.05,0,25,28.6


## 8 · Sellers: recompute aggregates & health-score conflicts

In [10]:
s = sellers
o_s = orders.groupby("seller_id")
s = s.rename(columns={"total_listings":"total_listings_reported", "total_sales_inr":"total_sales_reported_inr"})
s["listings_actual"] = s.seller_id.map(products.groupby("seller_id").size()).fillna(0).astype(int)
s["orders_actual"] = s.seller_id.map(o_s.size()).fillna(0).astype(int)
s["revenue_actual_inr"] = s.seller_id.map(orders[orders.is_cancelled == 0].groupby("seller_id").order_total_inr.sum()).fillna(0).round(2)
s["return_rate_actual_pct"] = (s.seller_id.map(o_s.is_returned.mean()).fillna(0) * 100).round(2)
note("sellers", "total_listings / total_sales_inr don't match products/orders tables (all 75 rows differ)", len(s),
     "Recomputed listings_actual, orders_actual, revenue_actual_inr, return_rate_actual_pct; originals kept as *_reported", "High")
s["has_orders"] = (s.orders_actual > 0).astype(int)
note("sellers", "Sellers with zero orders in period", (s.has_orders == 0).sum(), "Flagged has_orders = 0", "Low")

s["critical_but_high_rating"] = ((s.account_health == "Critical") & (s.seller_rating > 4.5)).astype(int)
s["healthy_but_high_risk"] = (((s.return_rate_pct + s.defect_rate_pct) > 15) & (s.account_health == "Healthy")).astype(int)
note("sellers", "account_health = Critical but seller_rating > 4.5", s.critical_but_high_rating.sum(), "Flagged", "Low")
note("sellers", "account_health = Healthy but return_rate + defect_rate > 15%", s.healthy_but_high_risk.sum(), "Flagged healthy_but_high_risk", "Medium")
print("Correlation reported vs actual return rate:", round(np.corrcoef(s.return_rate_pct, s.return_rate_actual_pct)[0,1], 3))
frames["sellers"] = sellers = s
s[["seller_id","total_listings_reported","listings_actual","total_sales_reported_inr","revenue_actual_inr"]].head()

Correlation reported vs actual return rate: -0.067


,seller_id,total_listings_reported,listings_actual,total_sales_reported_inr,revenue_actual_inr
0,INSLR0001,1592,0,16539246,0.00
1,INSLR0002,1649,3,5832393,467583.94
2,INSLR0003,4781,0,27516813,0.00
3,INSLR0004,762,3,36046568,58468.29
4,INSLR0005,1333,3,6168616,160207.36


## 9 · Reviews

In [11]:
rv_ = reviews.merge(orders[["order_id","delivery_date","order_status"]], on="order_id", how="left")
reviews["review_before_delivery"] = (rv_.review_date < rv_.delivery_date).astype(int).values
note("reviews", "Review dated before delivery", reviews.review_before_delivery.sum(), "Flagged review_before_delivery", "Low")
note("reviews", "helpful_votes > total_votes", (reviews.helpful_votes > reviews.total_votes).sum(), "None found", "Info")
note("reviews", "Verified purchase on cancelled order", ((reviews.verified_purchase == 1) & (rv_.order_status == "Cancelled").values).sum(), "None found", "Info")
note("reviews", "Review dated before order", (rv_.review_date < rv_.order_id.map(orders.set_index('order_id').order_date)).sum(), "None found", "Info")
print("Reviews only exist for orders with status:", rv_.order_status.unique())
print("Reviews per order (max):", reviews.order_id.value_counts().max(), "→ multiple reviews per order exist")
note("reviews", "Up to 5 reviews per single order (one order = one product)", (reviews.order_id.value_counts() > 1).sum(), "Flagged for review; kept", "Low")

Reviews only exist for orders with status: <StringArray>
['Delivered']
Length: 1, dtype: str
Reviews per order (max): 5 → multiple reviews per order exist


## 10 · Outlier scan (kept, not removed)

In [12]:
def iqr_outliers(s):
    q1, q3 = s.quantile([.25, .75]); return int(((s < q1 - 1.5*(q3-q1)) | (s > q3 + 1.5*(q3-q1))).sum())
for col in ["unit_price_inr","order_total_inr"]:
    n = iqr_outliers(orders[col]); print(f"orders.{col}: {n} IQR outliers ({n/len(orders):.1%})")
    note("orders", f"{col} IQR outliers (high-value electronics/mobiles)", n, "Kept: legitimate values, not errors; segment by category in EDA", "Info")
print("quantity range:", orders.quantity.min(), "-", orders.quantity.max(), "(no bulk-order outliers)")

orders.unit_price_inr: 307 IQR outliers (14.6%)
orders.order_total_inr: 283 IQR outliers (13.5%)
quantity range: 1 - 5 (no bulk-order outliers)


## 11 · Validation before export
Referential integrity, key uniqueness and no lost rows.

In [13]:
frames.update(sellers=sellers, customers=customers, products=products, orders=orders, returns=returns, transactions=transactions, reviews=reviews)
fks = [("products","seller_id","sellers","seller_id"), ("orders","customer_id","customers","customer_id"),
       ("orders","product_id","products","product_id"), ("orders","seller_id","sellers","seller_id"),
       ("returns","order_id","orders","order_id"), ("returns","customer_id","customers","customer_id"), ("returns","product_id","products","product_id"),
       ("transactions","order_id","orders","order_id"), ("transactions","customer_id","customers","customer_id"), ("transactions","seller_id","sellers","seller_id"),
       ("reviews","order_id","orders","order_id"), ("reviews","product_id","products","product_id"), ("reviews","customer_id","customers","customer_id")]
rows = []
for ct, cc, pt, pc in fks:
    orphans = (~frames[ct][cc].isin(frames[pt][pc])).sum(); rows.append((f"{ct}.{cc} → {pt}.{pc}", orphans)); assert orphans == 0
print(pd.DataFrame(rows, columns=["foreign key","orphan rows"]).to_string(index=False))
for t in TABLES:
    assert len(frames[t]) == len(raw[t]), t
    assert frames[t][pks[t]].is_unique
print("\nAll FK checks pass · PKs unique · row counts unchanged:", {t: len(frames[t]) for t in TABLES})

                                     foreign key  orphan rows
          products.seller_id → sellers.seller_id            0
      orders.customer_id → customers.customer_id            0
         orders.product_id → products.product_id            0
            orders.seller_id → sellers.seller_id            0
              returns.order_id → orders.order_id            0
     returns.customer_id → customers.customer_id            0
        returns.product_id → products.product_id            0
         transactions.order_id → orders.order_id            0
transactions.customer_id → customers.customer_id            0
      transactions.seller_id → sellers.seller_id            0
              reviews.order_id → orders.order_id            0
        reviews.product_id → products.product_id            0
     reviews.customer_id → customers.customer_id            0

All FK checks pass · PKs unique · row counts unchanged: {'sellers': 75, 'customers': 320, 'products': 180, 'orders': 2100, 'returns

## 12 · Export

In [14]:
for t in TABLES:
    frames[t].to_csv(CLEAN/f"{t}.csv", index=False, encoding="utf-8")

# data-quality log
dq = pd.DataFrame(log)
dq["severity"] = pd.Categorical(dq.severity, ["High","Medium","Low","Info"], ordered=True)
dq = dq.sort_values(["severity","rows_affected"], ascending=[True, False]).reset_index(drop=True)
dq.to_csv(REPORTS/"data_quality_log.csv", index=False)

# column-change catalogue (feeds the SQL Server schema design)
changes = []
for t in TABLES:
    old, new = set(raw[t].columns), set(frames[t].columns)
    for c in frames[t].columns:
        if c not in old: changes.append((t, c, "added"))
    for c in raw[t].columns:
        if c not in new: changes.append((t, c, "removed / renamed"))
pd.DataFrame(changes, columns=["table","column","change"]).to_csv(REPORTS/"column_changes.csv", index=False)

print("Issues logged by severity:"); print(dq[dq.rows_affected > 0].severity.value_counts().sort_index().to_string())
dq[dq.rows_affected > 0][["severity","table","issue","rows_affected","action"]]

Issues logged by severity:
severity
High       6
Medium    16
Low       10
Info       2


,severity,table,issue,rows_affected,action
0,High,transactions,Payment status contradicts order/return status,401,Flagged payment_conflict + payment_conflict_ty...
1,High,orders,Return records on orders whose status is 'Deli...,375,Kept raw status; added has_return_record / is_...
2,High,customers,total_orders / lifetime_spend_inr don't match ...,320,Recomputed orders_actual & lifetime_spend_actu...
3,High,transactions,Refund transactions (is_refund=1) with NULL li...,114,Attempted recovery via order_id join: 0 recove...
4,High,orders,Orders with status 'Returned' but NO return re...,106,Flagged returned_status_no_record; counted as ...
5,High,sellers,total_listings / total_sales_inr don't match p...,75,"Recomputed listings_actual, orders_actual, rev..."
6,Medium,orders,Promo code recorded but discount = 0,671,Flagged promo_without_discount
7,Medium,returns,days_to_return is measured from ORDER date (no...,420,Renamed to days_from_order_to_return; added da...
8,Medium,transactions,Transactions linked to a return but is_refund = 0,375,"Kept; is_refund is the payment-side flag, retu..."
9,Medium,orders,Discount given but no promo code,360,Flagged discount_without_promo


## Summary of decisions (for the README / interview)

| # | Issue | Decision |
|---|---|---|
| 1 | Order status vs returns table disagree | `returns` = source of truth; derived `is_returned`, `return_outcome`, `net_revenue_inr`; raw status kept |
| 2 | Payment status contradicts order status | Labelled with `payment_conflict_type`; kept for reconciliation analysis |
| 3 | Pre-aggregated columns (customers, sellers, products) don't match transactions | Recomputed from fact tables; originals kept as `*_reported` |
| 4 | `days_to_return` counted from order, not delivery | Added `days_since_delivery`; flagged window violations |
| 5 | `shipping_type` polluted with a status value | `shipping_type_clean` |
| 6 | Date/type issues, phone format, Yes/No text | Standardised (SQL Server-ready) |
| 7 | Logical oddities (orders before launch, negative margin, ...) | Flagged, not deleted |

**Next:** `02_eda.ipynb` and the SQL Server star schema (uses `reports/column_changes.csv`).